# EVE 310 - Lab 09: Batch processing — one building

**Module 3 | 10/22/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab09-batch-processing/notebooks/lab09-single-file.ipynb)

## Learning objectives

By the end of this lab you will be able to:

1. Load one campus water file and parse datetimes
2. Remove 3-sigma outliers
3. Draw a monthly box plot and save it

## Before you start

1. Click **Copy to Drive** at the top of this window and work in the copy that opens. Colab throws away anything you did not copy when the runtime ends.
2. Run the setup cell below before anything else. It creates `DATA_DIR` and `FIGURES_DIR` and downloads this lab's data files.
3. Work down the notebook in order. Cells marked **Your turn** are the ones you complete.
4. Nothing to submit for this notebook. The **activity** notebook is the one you download as `.ipynb` and upload to Gradescope.


Campus building water files live in `DATA_DIR` with names like `water_DCP.csv`. The three-letter code after `water_` is the building.


## 0. Setup


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# EVE 310 setup - run this cell first, every time you open this notebook.
# It downloads this lab's data files into a "data" folder in your Colab session.
import pathlib
import urllib.request

LAB = "lab09-batch-processing"
DATA_FILES = [f"water_{code}.csv" for code in "AHG AND ARC ART BAT BEN BHD BLD BMA BME BUR CAL CBA CDL CLA CMA CPE CRD CRH DCP DFA EAS ECj EPS".split()]

DATA_DIR = pathlib.Path("data")
FIGURES_DIR = pathlib.Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

BASE_URL = f"https://raw.githubusercontent.com/ThyanRevolter/eve310-fall-2026/main/labs/{LAB}/data"
for name in DATA_FILES:
    if not (DATA_DIR / name).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", DATA_DIR / name)

print(f"Ready. {len(DATA_FILES)} data file(s) in {DATA_DIR.resolve()}")

## 1. Load one file


In [ ]:
file_path = DATA_DIR / 'water_DCP.csv'
water_df = pd.read_csv(file_path)
building = file_path.stem.replace('water_', '')
print(building)
water_df.head()


## 2. Worked example


In [ ]:
water_df['DateTime'] = pd.to_datetime(water_df['DateTime'])
water_df['Month'] = water_df['DateTime'].dt.month
col = 'Water ( Gallons )'

w_std = np.std(water_df[col], ddof=1)
w_mean = np.mean(water_df[col])
upper, lower = w_mean + 3 * w_std, w_mean - 3 * w_std
water_df = water_df.loc[(water_df[col] < upper) & (water_df[col] > lower)]

ax = water_df.boxplot(column=col, by='Month', grid=False, rot=45)
plt.xticks(
    range(1, 13),
    ['January', 'February', 'March', 'April', 'May', 'June',
     'July', 'August', 'September', 'October', 'November', 'December'],
)
plt.ylabel('Consumption (gallons)')
plt.xlabel('Month')
plt.title(f'{building} water consumption by month')
plt.suptitle('')
plt.savefig(FIGURES_DIR / f'{building}_water_boxplot.png', bbox_inches='tight', dpi=200)


## 3. Wrap-up

Next: loop over every CSV in `DATA_DIR` in `lab09-multiple-files.ipynb`.
